In [ ]:
from thefuzz import process
import cn2an
import logging
import re
from utils import pre_process, pre_process_without_n

In [ ]:
def find_权宜处理_rule_base(docs,一般政策性内容):
    """Return the sentence that match 权宜处理 keywords using regular expression and fuzzy matching

    Returns:
        list of tuple: the all sentences that match 权宜处理 keywords and the matched keywords
        list of tuple: the index of the matched sentences in the original text and the matched keywords
    """
    # preprocess the 一般政策性内容 to remove any blank space
    一般政策性内容 = [re.sub(r'\s+', '', content) for content in 一般政策性内容]
    
    logging.getLogger().setLevel(logging.ERROR)
    
    # keywords for regular expression
    keywords = ["结合.*?实际", "根据.*?实际", "根据实际情况", "结合实际情况", "结合本地实际", "根据本地实际","因地制宜"]
    pattern = re.compile('|'.join(keywords))
    
    # keywords for fuzzy matching
    keywords_fuzzy = ["结合实际", "根据实际", "根据实际情况", "结合实际情况", "结合本地实际", "根据本地实际"]
    
    paragraphs = pre_process(docs)
    
    matched_paragraphs = []

    for paragraph in paragraphs:
        # remove any blank space before and after the paragraph
        check_paragraph = re.sub(r'\s+', '', paragraph)
        if check_paragraph in 一般政策性内容:
            continue
        elif pattern.search(paragraph):
            matched_paragraphs.append((paragraph.strip(), pattern.search(paragraph).group()))
        else:
            best_match = process.extractOne(paragraph, keywords_fuzzy)
            if best_match[1] >= 60:  
                matched_paragraphs.append((paragraph.strip(), best_match[0]))

    matched_paragraphs_index = []
    for sentence in matched_paragraphs:
        begin_index = docs.find(sentence[0])
        end_index = begin_index + len(sentence[0])
        matched_paragraphs_index.append((begin_index, end_index))
        
    return matched_paragraphs, matched_paragraphs_index
